## BKT Training
Fit the existing four-parameter mastery model from scored assessment sequences.

In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import os

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/hlo_recommender").is_dir())
os.chdir(ROOT)

from hlo_recommender.cli import parser, run
from hlo_recommender.io import read_jsonl, write_json, write_jsonl

SESSION_FILE = ROOT / "artifacts/notebooks/session.json"
SESSION = json.loads(SESSION_FILE.read_text()) if SESSION_FILE.exists() else {}


In [ ]:
OUTPUT = ROOT / "artifacts/notebooks" / f"bkt-{uuid4().hex[:8]}"
CONFIG_SOURCE = "configs/bkt.development.toml"
PATH_FIELDS = ("train", "validation")
TRAIN_FILE = "assessments.train.jsonl"
VALIDATION_FILE = "assessments.validation.jsonl"

import tomllib

settings = tomllib.loads((ROOT / CONFIG_SOURCE).read_text())
for key in PATH_FIELDS:
    settings[key] = str(ROOT / settings[key])
DATA = Path(SESSION.get("data_directory", ROOT / "data/development"))
settings["train"] = str(DATA / TRAIN_FILE)
settings["validation"] = str(DATA / VALIDATION_FILE)
settings["output"] = str(OUTPUT)
CONFIG_PATH = ROOT / "artifacts/notebooks/configs" / f"{OUTPUT.name}.toml"
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text("\n".join(f"{key} = {json.dumps(value)}" for key, value in settings.items()) + "\n")
settings


### Fitting
Use the repository maximum-likelihood fitting procedure with learner-disjoint validation.

In [ ]:
trained = run(parser().parse_args(["train", "bkt", "--config", str(CONFIG_PATH), "--allow-synthetic"]))
SESSION["bkt_model"] = str(OUTPUT / "model.json")
write_json(SESSION_FILE, SESSION)
{"parameters": trained["parameters"], "fit": trained["fit"], "validation": trained["validation"]}


### Mastery Update
Apply the same trained parameters to correct and incorrect responses.

In [ ]:
from hlo_recommender.models.bkt import load_parameters

parameters = load_parameters(SESSION["bkt_model"])
{"prior": parameters.initial, "correct": parameters.update(parameters.initial, True), "incorrect": parameters.update(parameters.initial, False)}
